In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_predict, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, roc_curve, auc, confusion_matrix,
                             precision_recall_curve, average_precision_score, classification_report)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
import joblib
import warnings
import os
from collections import Counter
from scipy.stats import randint, uniform, wilcoxon
from imblearn.pipeline import Pipeline as ImbPipeline
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
File1 = pd.read_csv('C:/Users/User/Downloads/canada_water_pollution.csv')
File1.head()

,Sample_ID,Province,City,Date_Collected,Water_Source,Pollutant_Type,Pollutant_Level_mg_L,Safe_Limit_mg_L,Exceeds_Limit,pH_Level,Temperature_C,Turbidity_NTU,Conductivity_uS_cm,Dissolved_Oxygen_mg_L,Treatment_Status,Agency_Responsible,Latitude,Longitude,Sampling_Depth_m,Comments
0,SMP100000,Quebec,Port Stephentown,2023-12-30,Coastal,Microplastics,35.44,23.94,True,8.89,14.4,54.90,187.18,10.90,Untreated,WaterSafe Canada,53.336879,-95.612118,3.13,Teacher prevent current any.
1,SMP100001,Alberta,Deanshire,2024-05-02,Groundwater,Phosphate,24.20,14.71,True,7.98,25.2,76.54,1525.52,12.31,Partially Treated,WaterSafe Canada,47.781417,-59.407985,4.89,Industry realize deep nothing.
2,SMP100002,British Columbia,West Catherineside,2023-03-30,River,Pathogens,13.59,19.48,False,7.73,18.9,78.58,1382.09,11.10,Untreated,CleanH2O Org,54.850576,-110.073418,3.20,Believe short how family some certain.
3,SMP100003,Ontario,Ryantown,2023-12-21,Lake,Mercury,9.67,26.01,False,6.80,12.6,33.37,1577.37,7.70,Partially Treated,HydroWatch,49.920359,-81.200860,1.39,Fly buy politics image network.
4,SMP100004,Alberta,West Lisaside,2024-04-27,Groundwater,Pathogens,30.67,4.83,True,5.44,2.5,0.97,472.16,9.39,Partially Treated,WaterSafe Canada,56.612537,-112.156988,5.15,Mr work strategy western structure beat.


In [4]:
columns = ['Temperature_C', 'pH_Level', 'Turbidity_NTU', 'Conductivity_uS_cm']
df = File1[columns]
			
df
#df[(df['Pollution_Level'] == "Poor")].head()

,Temperature_C,pH_Level,Turbidity_NTU,Conductivity_uS_cm
0,14.4,8.89,54.90,187.18
1,25.2,7.98,76.54,1525.52
2,18.9,7.73,78.58,1382.09
3,12.6,6.80,33.37,1577.37
4,2.5,5.44,0.97,472.16
...,...,...,...,...
2995,27.3,7.12,85.38,1061.81
2996,16.1,8.94,78.62,568.19
2997,9.1,6.07,6.86,1432.11
2998,16.0,7.43,20.81,1880.94


In [5]:

File2 = pd.read_csv('C:/Users/User/Downloads/Water Quality Prediction.csv', encoding='latin1')
File2 = File2.dropna()
#File2 = File2.sample(n=17000, random_state=200)

print(File2.shape)


(701056, 24)


In [6]:
def get_nlwqs_class(ph, turb, cond, temp, normal_temp=31.0):

    # Category A
    if (6.5 <= ph <= 8.5) and (cond <= 1000) and (temp <= normal_temp) and (turb <= 40):
        return 1
    
    # Category B
    elif (6.5 <= ph <= 8.5) and (cond <= 1000) and (temp <= normal_temp) and (40 <= turb <= 170):
        return 2
    
    # Category C
    elif (6.0 <= ph <= 9.0) and (cond <= 2000) and (temp <= normal_temp) and (turb <= 70):
        return 3
    
    # Category D
    elif (5.5 <= ph <= 9.0) and (cond <= 5000) and (temp <= normal_temp) and (turb <= 250):
        return 4
    
    else:
        return 5
    

In [7]:
columns2 = ['Water Temperature', 'pH', 'Turbidity', 'Conductivity']
df2 = File2[columns2]
df2['overall_class_nlwqs'] = df2.apply(lambda row: get_nlwqs_class(row['pH'], row['Turbidity'], row['Conductivity'], row['Water Temperature']), axis=1)
overall_class_counts = df2['overall_class_nlwqs'].value_counts()
display(overall_class_counts)

C:\Users\User\AppData\Local\Temp\ipykernel_13748\1802041407.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['overall_class_nlwqs'] = df2.apply(lambda row: get_nlwqs_class(row['pH'], row['Turbidity'], row['Conductivity'], row['Water Temperature']), axis=1)


overall_class_nlwqs
1    518478
5    120439
3     47655
4     14484
Name: count, dtype: int64

In [8]:
target_size = 3000
dfs = []

for cls in df2['overall_class_nlwqs'].unique():
    cls_df = df2[df2['overall_class_nlwqs'] == cls]
    dfs.append(cls_df.sample(n=min(target_size, len(cls_df)), random_state=42))

df_balanced = pd.concat(dfs).sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanced.shape)
print(df_balanced['overall_class_nlwqs'].value_counts())

df2 = df_balanced


(12000, 5)
overall_class_nlwqs
1    3000
3    3000
4    3000
5    3000
Name: count, dtype: int64


In [9]:
df['overall_class_nlwqs'] = df.apply(lambda row: get_nlwqs_class(row['pH_Level'], row['Turbidity_NTU'], row['Conductivity_uS_cm'], row['Temperature_C']), axis=1)
df

C:\Users\User\AppData\Local\Temp\ipykernel_13748\4223187064.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['overall_class_nlwqs'] = df.apply(lambda row: get_nlwqs_class(row['pH_Level'], row['Turbidity_NTU'], row['Conductivity_uS_cm'], row['Temperature_C']), axis=1)


,Temperature_C,pH_Level,Turbidity_NTU,Conductivity_uS_cm,overall_class_nlwqs
0,14.4,8.89,54.90,187.18,3
1,25.2,7.98,76.54,1525.52,4
2,18.9,7.73,78.58,1382.09,4
3,12.6,6.80,33.37,1577.37,3
4,2.5,5.44,0.97,472.16,5
...,...,...,...,...,...
2995,27.3,7.12,85.38,1061.81,4
2996,16.1,8.94,78.62,568.19,4
2997,9.1,6.07,6.86,1432.11,3
2998,16.0,7.43,20.81,1880.94,3


In [10]:
df2 = df2.rename(columns={
    'Water Temperature': 'Temperature_C',
    'pH': 'pH_Level',
    'Turbidity': 'Turbidity_NTU',
    'Conductivity': 'Conductivity_uS_cm'
})

df2

,Temperature_C,pH_Level,Turbidity_NTU,Conductivity_uS_cm,overall_class_nlwqs
0,12.998176,7.841381,4.369301,549.468246,1
1,7.063896,6.447448,0.595614,118.414472,3
2,18.669596,7.116749,0.188914,825.668225,1
3,18.966545,5.604543,0.119007,451.453394,4
4,23.608063,7.687311,0.000007,283.223480,1
...,...,...,...,...,...
11995,24.517882,5.957864,0.055267,495.199870,4
11996,15.306271,9.865261,0.003115,328.103279,5
11997,20.946305,9.546205,2.793094,395.130512,5
11998,15.673838,7.680605,0.144682,472.942964,1


In [11]:
combined_df = pd.concat([df, df2], ignore_index=True)
print(combined_df.shape)

(15000, 5)


In [12]:
overall_class_counts = combined_df['overall_class_nlwqs'].value_counts()
display(overall_class_counts)

overall_class_nlwqs
3    4082
4    3814
5    3366
1    3275
2     463
Name: count, dtype: int64

In [13]:
df = combined_df

In [14]:
features = ['Conductivity_uS_cm', 'pH_Level', 'Turbidity_NTU', 'Temperature_C']
X = df[features].values
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['overall_class_nlwqs'])
unique, counts = np.unique(y, return_counts=True)

In [15]:
warnings.filterwarnings('ignore')

n_folds = 5
n_iter = 20
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

random_forest = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced')
random_forest_param = { 'classifier__n_estimators': randint(50, 200), 'classifier__max_depth': [5, 10, 15, None], 'classifier__min_samples_split': randint(2, 15), 'classifier__min_samples_leaf': randint(1, 5)}

xgboost = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1)
xgboost_param = { 'classifier__n_estimators': randint(50, 200), 'classifier__max_depth': randint(3, 8), 'classifier__learning_rate': uniform(0.01, 0.2)}

svm = SVC(probability=True, random_state=42,class_weight='balanced')
svm_param = { 'classifier__C': uniform(0.1, 10), 'classifier__gamma': ['scale', 'auto'], 'classifier__kernel': ['rbf']}

knn = KNeighborsClassifier(n_jobs=-1)
knn_param = { 'classifier__n_neighbors': randint(3, 15), 'classifier__weights': ['uniform', 'distance'] }

mlp_shallow = MLPClassifier(random_state=42, max_iter=300, early_stopping=True)
mlp_shallow_param = { 'classifier__hidden_layer_sizes': [(50,), (100,), (50, 25)], 'classifier__activation': ['relu', 'tanh'], 'classifier__alpha': uniform(0.0001, 0.01) }

mlp_deep = MLPClassifier(random_state=42, max_iter=300, early_stopping=True)
mlp_deep_param = { 'classifier__hidden_layer_sizes': [(128, 64, 32), (256, 128, 64)], 'classifier__activation': ['relu'], 'classifier__alpha': uniform(0.001, 0.05) }

models = ['Random Forest', 'XGBoost', 'SVM', 'KNN', 'MLP (Shallow)', 'DNN (Deep)']
model_objects = [random_forest, xgboost, svm, knn, mlp_shallow, mlp_deep]
model_parameters = [random_forest_param, xgboost_param, svm_param, knn_param, mlp_shallow_param, mlp_deep_param]

In [17]:
best_models = {}

for i in range(len(models)):
   print("\nTraining " + models[i])
   pipeline = ImbPipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=42)), ('classifier', model_objects[i])])
   random_search = RandomizedSearchCV(
        estimator=pipeline, 
        param_distributions=model_parameters[i], 
        n_iter=n_iter, 
        cv=skf, 
        scoring='f1_weighted', 
        random_state=42, 
        n_jobs=-1, 
        verbose=0
    )
   
   random_search.fit(X, y)
   best_models[models[i]] = random_search.best_estimator_
   y_pred_cv = cross_val_predict(best_models[models[i]], X, y, cv=skf)
   y_proba_cv = cross_val_predict(best_models[models[i]], X, y, cv=skf, method='predict_proba')
   acc = accuracy_score(y, y_pred_cv)
   prec = precision_score(y, y_pred_cv, average='weighted', zero_division=0)
   rec = recall_score(y, y_pred_cv, average='weighted', zero_division=0)
   f1 = f1_score(y, y_pred_cv, average='weighted', zero_division=0)
   auc_score = roc_auc_score(y, y_proba_cv, multi_class='ovr', average='weighted')
   print(f"Best params: {random_search.best_params_}")
   print(f"Accuracy: {acc}")
   print(f"Precision: {prec}")
   print(f"Recall: {rec}")
   print(f"F1-Score: {f1}")
   print(f"AUC-ROC: {auc_score}")

save_dir = "dataset_1_classification"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

ensemble_package = {}
ensemble_package['models'] = best_models
ensemble_package['features'] = features
ensemble_package['encoder'] = label_encoder
unified_path = f"{save_dir}/dataset_1_classification.pkl"
joblib.dump(ensemble_package, unified_path)
print("Saved to " + unified_path)


Training Random Forest
Best params: {'classifier__max_depth': 10, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 4, 'classifier__n_estimators': 157}
Accuracy: 0.9995333333333334
Precision: 0.9995335360983292
Recall: 0.9995333333333334
F1-Score: 0.9995332368268878
AUC-ROC: 0.9999981619585995

Training XGBoost
Best params: {'classifier__learning_rate': np.float64(0.20312640661491188), 'classifier__max_depth': 4, 'classifier__n_estimators': 58}
Accuracy: 0.9924666666666667
Precision: 0.9924726905412283
Recall: 0.9924666666666667
F1-Score: 0.9924678438926335
AUC-ROC: 0.9999337865508305

Training SVM
Best params: {'classifier__C': np.float64(9.83755518841459), 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}
Accuracy: 0.9729333333333333
Precision: 0.9736410599549328
Recall: 0.9729333333333333
F1-Score: 0.9730718433927504
AUC-ROC: 0.9991093283172555

Training KNN
Best params: {'classifier__n_neighbors': 3, 'classifier__weights': 'distance'}
Accuracy: 0.9366666

In [18]:
warnings.filterwarnings('ignore')

anomaly_labels = []
for value in df['overall_class_nlwqs']:
    if value == 5:
        anomaly_labels.append(1)
    else:
        anomaly_labels.append(0)

df['is_anomaly'] = anomaly_labels
y = df['is_anomaly'].values

In [19]:
n_folds = 10
n_iter = 30
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)
random_forest_param = { 'classifier__n_estimators': randint(50, 300), 'classifier__max_depth': [5, 10, 15, None], 'classifier__min_samples_split': randint(2, 20), 'classifier__min_samples_leaf': randint(1, 10), 'classifier__max_features': ['sqrt', 'log2', None]}

xgboost = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss', n_jobs=-1)
xgboost_param = { 'classifier__n_estimators': randint(50, 300), 'classifier__max_depth': randint(3, 10), 'classifier__learning_rate': uniform(0.01, 0.3), 'classifier__subsample': uniform(0.6, 0.4), 'classifier__colsample_bytree': uniform(0.6, 0.4)}

svm = SVC(probability=True, random_state=42)
svm_param = { 'classifier__C': uniform(0.1, 10), 'classifier__gamma': ['scale', 'auto'], 'classifier__kernel': ['rbf']}

knn = KNeighborsClassifier(n_jobs=-1)
knn_param = { 'classifier__n_neighbors': randint(3, 15), 'classifier__weights': ['uniform', 'distance'], 'classifier__p': [1, 2]}

mlp_shallow = MLPClassifier(random_state=42, max_iter=500, early_stopping=True)
mlp_shallow_param = { 'classifier__hidden_layer_sizes': [(50,), (100,)], 'classifier__activation': ['relu', 'tanh'], 'classifier__alpha': uniform(0.0001, 0.01) }

mlp_deep = MLPClassifier(random_state=42, max_iter=500, early_stopping=True)
mlp_deep_param = { 'classifier__hidden_layer_sizes': [(128, 64, 32), (256, 128, 64)], 'classifier__activation': ['relu'], 'classifier__alpha': uniform(0.001, 0.01) }

models1 = ['Random Forest', 'XGBoost', 'SVM', 'KNN', 'MLP (Shallow)', 'DNN (Deep)']
model_objects1 = [random_forest, xgboost, svm, knn, mlp_shallow, mlp_deep]
model_parameters1 = [random_forest_param, xgboost_param, svm_param, knn_param, mlp_shallow_param, mlp_deep_param]

In [20]:
best_models1 = {}
all_f1_scores = {}
all_metrics = {}
print("\nStarting anomaly detection training...")
for i in range(len(models1)):
   print("\nTraining " + models1[i])
   pipeline = ImbPipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=42)), ('classifier', model_objects1[i])])
   random_search = RandomizedSearchCV(
        estimator=pipeline, 
        param_distributions=model_parameters1[i], 
        n_iter=n_iter, 
        cv=skf, 
        scoring='f1', 
        random_state=42, 
        n_jobs=-1, 
        verbose=0
    )
   
   random_search.fit(X, y)
   best_models1[models1[i]] = random_search.best_estimator_
   fold_scores = cross_val_score(random_search.best_estimator_, X, y, cv=skf, scoring='f1')
   all_f1_scores[models1[i]] = fold_scores
   
   y_pred_cv = cross_val_predict(best_models1[models1[i]], X, y, cv=skf, method='predict')
   y_proba_cv = cross_val_predict(best_models1[models1[i]], X, y, cv=skf, method='predict_proba')

   y_proba_positive = y_proba_cv[:, 1]
   acc = accuracy_score(y, y_pred_cv)
   prec = precision_score(y, y_pred_cv, zero_division=0)
   rec = recall_score(y, y_pred_cv, zero_division=0)
   f1 = f1_score(y, y_pred_cv, zero_division=0)
   f1_std = fold_scores.std()
   roc_auc = roc_auc_score(y,  y_proba_positive)
   pr_auc = average_precision_score(y, y_proba_positive)
   all_metrics[models1[i]] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'F1-Std': f1_std,
        'ROC-AUC': roc_auc,
        'PR-AUC': pr_auc
    }
   
   print(f"{models1[i]} Finished")
   print(all_metrics[models1[i]])

save_dir = "dataset_1_anomaly"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

ensemble_package = {}
ensemble_package['models'] = best_models1
ensemble_package['features'] = features
ensemble_package['encoder'] = label_encoder
unified_path = save_dir + "/dataset_1_anomaly.pkl"
joblib.dump(ensemble_package, unified_path)
print("Saved to " + unified_path)


Starting anomaly detection training...

Training Random Forest
Random Forest Finished
{'Accuracy': 0.9998, 'Precision': 0.999702823179792, 'Recall': 0.9994058229352347, 'F1-Score': 0.9995543009953944, 'F1-Std': np.float64(0.0006809245092923336), 'ROC-AUC': 0.9999994126666456, 'PR-AUC': 0.9999979825920897}

Training XGBoost
XGBoost Finished
{'Accuracy': 0.9974, 'Precision': 0.9952366775826139, 'Recall': 0.9931669637551991, 'F1-Score': 0.9942007434944238, 'F1-Std': np.float64(0.0030858312882337505), 'ROC-AUC': 0.9999665603031498, 'PR-AUC': 0.9998856242472429}

Training SVM
SVM Finished
{'Accuracy': 0.9935333333333334, 'Precision': 0.9800293685756241, 'Recall': 0.9913844325609031, 'F1-Score': 0.9856741987889529, 'F1-Std': np.float64(0.0048191991704220755), 'ROC-AUC': 0.9998015323986869, 'PR-AUC': 0.9993427908671941}

Training KNN
KNN Finished
{'Accuracy': 0.9794, 'Precision': 0.9416353655013002, 'Recall': 0.9682115270350564, 'F1-Score': 0.9547385381573166, 'F1-Std': np.float64(0.00850727